# AICE Associate 변주 문제 - 부동산 등급,고가_저가 예측 (분류)

### 시나리오

AICE 부동산 컨설팅 회사는 잠재 투자자들에게 특정 지역의 부동산 매입 적합성을 평가하기 위해 모델을 구축하고자 합니다. 이들은 단순히 가격을 예측하는 것을 넘어, 해당 부동산이 시장에서 '고가(Premium)' 또는 '일반(Standard)' 등급으로 분류될 확률을 예측하여 투자 포트폴리오를 최적화하려고 합니다. 이 분류는 위험도 분석과 투자 우선순위를 결정하는 데 필수적입니다.

---

**[유의사항]**
- 답안은 각 문항 아래 표시된 `# (N) 여기에 ...` 칸에 작성하세요.
- **정답/해설은 이 노트북 가장 아래 `## 해설` 섹션에 모아뒀습니다.** 먼저 스스로 풀어본 뒤에 확인하세요.
- 이 노트북은 오리지널 창작 문제이며, 실제 AICE 샘플문항 원문을 복제하지 않습니다.

**[데이터 컬럼 설명]**

- Property_Class : 부동산 등급,고가_저가
- Sale_Price_KRW : 거래된 부동산의 최종 가격 (단위: 원)
- District_Type : 지역 유형,도심형,교외형,산업형
- Area_SqM : 피처 컬럼
- Age_Years : 피처 컬럼
- Floor_Type : 피처 컬럼
- Commercial_Density : 피처 컬럼
- Year_Built : 피처 컬럼
- Property_ID : 식별자(모델링에 불필요)

## 0. 데이터 준비

다음 문항을 풀기 전에 아래 코드를 실행하세요(문제 데이터를 생성합니다).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def synthesize_dataframe(seed=0, n_rows=600):
    rng = np.random.default_rng(seed)
    n = n_rows

    base = rng.normal(loc=50, scale=15, size=n)
    outlier_idx = rng.choice(n, size=max(3, n // 50), replace=False)
    base[outlier_idx] += rng.choice([1, -1], size=len(outlier_idx)) * rng.uniform(80, 150, size=len(outlier_idx))

    df = pd.DataFrame({'Sale_Price_KRW': base.round(2)})
    n_cats = rng.integers(3, 5)
    cats = [f"Type{i+1}" for i in range(n_cats)]
    df['District_Type'] = rng.choice(cats, size=n)

    for col in ['Area_SqM', 'Age_Years', 'Floor_Type', 'Commercial_Density', 'Year_Built']:
        df[col] = rng.normal(0, 1, size=n).round(2)

    df['Property_ID'] = [f"ID{i:05d}" for i in range(n)]

    z = (base - base.mean()) / (base.std() + 1e-9)
    if '분류' == "분류":
        prob = 1 / (1 + np.exp(-z))
        labels = (rng.random(n) < prob).astype(int)
        classes = [c.strip() for c in '부동산 등급,고가_저가'.split(",") if c.strip()][:2]
        if len(classes) < 2:
            classes = ["Positive", "Negative"]
        df['Property_Class'] = np.where(labels == 1, classes[0], classes[-1])
    else:
        noise = rng.normal(0, 5, size=n)
        df['Property_Class'] = (base * 1.5 + noise).round(2)

    for col in ['Sale_Price_KRW'] + ['Area_SqM', 'Age_Years', 'Floor_Type', 'Commercial_Density', 'Year_Built'][:2]:
        na_idx = rng.choice(n, size=int(n * 0.03), replace=False)
        df.loc[na_idx, col] = np.nan

    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


data = synthesize_dataframe(seed=1006)
data.to_csv("data.csv", index=False)
print("data.csv 저장 완료 -", data.shape)
data.head(4)

## <데이터 분석>

### 1. 라이브러리 임포트

scikit-learn 을 alias **sk** 로 임포트하세요.

In [ ]:
# (1) 여기에 답안코드를 작성하고 실행하세요



### 2. 데이터 로드

pandas 로 'data.csv' 를 읽어와 데이터프레임 변수명 **my_data** 에 할당하고, 첫 4개 행을 출력하세요.

In [ ]:
# (2) 여기에 답안코드를 작성하고 실행하세요



### 3. 시각화 (subplots)

지역 유형,도심형,교외형,산업형(District_Type) 분포의 countplot 과, 부동산 등급,고가_저가 별 거래된 부동산의 최종 가격 (단위: 원)(Sale_Price_KRW) histplot 을 나란히 그리는 코드입니다.
빈칸 **(A)** 에 들어갈, 여러 그래프를 한 번에 그릴 때 쓰는 matplotlib 함수 이름은?

```python
fig, axes = plt.(A)(nrows=1, ncols=2, figsize=(12, 5))
sns.countplot(data=my_data, x='District_Type', ax=axes[0])
sns.histplot(data=my_data, x='Sale_Price_KRW', hue='Property_Class', ax=axes[1])
plt.show()
```

In [ ]:
# (3) 여기에 답을 입력하세요 (실행 불필요)



### 4. 시각화 (boxplot)

부동산 등급,고가_저가 별 거래된 부동산의 최종 가격 (단위: 원)(Sale_Price_KRW) 의 분포를 seaborn boxplot 으로 시각화하세요 (X축: 타깃, Y축: 수치형 컬럼).

In [ ]:
# (4) 여기에 답안코드를 작성하고 실행하세요



## <데이터 전처리>

### 5. 이상치 처리

거래된 부동산의 최종 가격 (단위: 원)(Sale_Price_KRW) 컬럼의 IQR 기준(K=1.0)을 벗어나는 이상치 행을 제거하고, 불필요한 컬럼(Property_ID)도 삭제해서 **data_temp** 에 저장하는 코드입니다. 빈칸 **(A)** 에 들어갈, 행을 삭제할 때 쓰는 DataFrame 메서드 이름은?

```python
q1 = my_data['Sale_Price_KRW'].quantile(0.25)
q3 = my_data['Sale_Price_KRW'].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.0 * iqr
upper_fence = q3 + 1.0 * iqr
data_temp = my_data.(A)(my_data[(my_data['Sale_Price_KRW'] > upper_fence) | (my_data['Sale_Price_KRW'] < lower_fence)].index)
data_temp = data_temp.drop(columns=['Property_ID'])
data_temp = data_temp.reset_index(drop=True)
```

In [ ]:
# (5) 여기에 답을 입력하세요 (실행 불필요)



### 6. 결측치 처리

다음은 data_temp 의 결측치를 처리하는 코드인데, 실행하면 의도한 것과 다르게 동작합니다. 무엇이 문제인지 서술하거나, 올바르게 고친 코드를 작성하세요.

```python
print('결측치 처리 전')
print(data_temp.isnull().sum())
data_na = data_temp.dropna()
print('결측치 처리 후')
print(data_na.isnull().sum())
# (이 버전에는 의도한 것과 다른 결과를 내는 부분이 있습니다)
```

In [ ]:
# (6) 여기에 문제점 설명 또는 수정한 코드를 작성하세요



### 7. 인코딩

지역 유형,도심형,교외형,산업형(District_Type) 컬럼을 sklearn 의 **LabelEncoder** 로 인코딩해서 data_preset 에 저장하세요.

In [ ]:
# (7) 여기에 답안코드를 작성하고 실행하세요



### 8. 데이터 분리

Property_Class 을 y, 나머지를 X 로 삼아 train_test_split 으로 분리하세요.
- test_size=0.2, random_state=7, stratify 옵션을 적용하세요
- 변수명: X_train, X_valid, y_train, y_valid

In [ ]:
# (8) 여기에 답안코드를 작성하고 실행하세요



### 9. 스케일링

MinMaxScaler 로 훈련/검증 데이터를 스케일링하는 코드인데, 의도한 것과 다르게 동작합니다. 무엇이 문제인지 서술하거나, 올바르게 고친 코드를 작성하세요.

```python
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
if len(X_train) > 0:
    X_train_scaled = scaler.fit_transform(X_train)
else:
    X_train_scaled = None
X_valid_scaled = scaler.transform(X_valid)
```

In [ ]:
# (9) 여기에 문제점 설명 또는 수정한 코드를 작성하세요



## <AI 모델링>

### 10. GridSearch 모델링

DecisionTreeClassifier 와 RandomForestClassifier 를 GridSearchCV(cv=3)로 탐색하고 학습하세요.
- max_depth 후보: [3, 5, 7]
- RandomForestClassifier 의 n_estimators 후보: [50, 100, 200]
- 변수명: gs_a (트리 모델), gs_b (앙상블 모델)

In [ ]:
# (10) 여기에 답안코드를 작성하고 실행하세요



### 11. GridSearch 결과 확인

위 GridSearch에서 RandomForestClassifier 의 n_estimators 후보는 [50, 100, 200] 였습니다. GridSearchCV가 고를 수 있는 값의 후보 중 '가장 큰 값'은 얼마인가요?

In [ ]:
# (11) 여기에 답을 입력하세요 (실행 불필요)



### 12. 변수중요도

RandomForestClassifier 의 변수중요도 Top 15개를 뽑아 시각화하는 코드인데, 의도한 것과 다르게 동작합니다. 무엇이 문제인지 서술하거나, 올바르게 고친 코드를 작성하세요.

```python
fi = pd.DataFrame({'feature': X_train.columns, 'importance': gs_b.best_estimator_.feature_importances_})
fi = fi.sort_values('importance', ascending=False)[:16]
sns.barplot(x='importance', y='feature', data=fi)
plt.show()
```

In [ ]:
# (12) 여기에 문제점 설명 또는 수정한 코드를 작성하세요



### 13. 성능평가

검증데이터로 gs_a, gs_b 두 모델의 **f1_score** 를 각각 계산해서 a_score, b_score 에 저장하세요 (타깃이 문자열 클래스이므로 average='macro' 를 함께 쓰세요).

In [ ]:
# (13) 여기에 답안코드를 작성하고 실행하세요



---
## 해설

스스로 풀어본 뒤 아래에서 확인하세요. 문항 번호가 위 문제 번호와 일치합니다.

### 1번 해설 - 라이브러리 임포트 [코드작성]

In [ ]:
import sklearn as sk

> import ... as ... 문법으로 별칭을 지정합니다.

### 2번 해설 - 데이터 로드 [코드작성]

In [ ]:
my_data = pd.read_csv('data.csv')
my_data.head(4)

> pd.read_csv() 로 파일을 읽고 head() 로 상위 행을 확인합니다.

### 3번 해설 - 시각화 (subplots) [빈칸채우기]

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 5))
sns.countplot(data=my_data, x='District_Type', ax=axes[0])
sns.histplot(data=my_data, x='Sale_Price_KRW', hue='Property_Class', ax=axes[1])
plt.show()

> plt.subplots() 는 nrows/ncols 로 여러 축(Axes)을 한 번에 만듭니다. 정답: subplots

### 4번 해설 - 시각화 (boxplot) [코드작성]

In [ ]:
sns.boxplot(data=my_data, x='Property_Class', y='Sale_Price_KRW')
plt.show()

> sns.boxplot(x=타깃, y=수치형컬럼) 형태로 그립니다.

### 5번 해설 - 이상치 처리 [빈칸채우기]

In [ ]:
q1 = my_data['Sale_Price_KRW'].quantile(0.25)
q3 = my_data['Sale_Price_KRW'].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.0 * iqr
upper_fence = q3 + 1.0 * iqr
data_temp = my_data.drop(my_data[(my_data['Sale_Price_KRW'] > upper_fence) | (my_data['Sale_Price_KRW'] < lower_fence)].index)
data_temp = data_temp.drop(columns=['Property_ID'])
data_temp = data_temp.reset_index(drop=True)

> DataFrame.drop() 은 행(기본 axis=0) 또는 열(axis=1)을 삭제합니다. 정답: drop

### 6번 해설 - 결측치 처리 [오류정정]

In [ ]:
print('결측치 처리 전')
print(data_temp.isnull().sum())
data_na = data_temp.dropna()
print('결측치 처리 후')
print(data_na.isnull().sum())

> [off_by_one] 오프바이원(Off-by-one) 오류: 인덱스 범위 초과, 반복/개수가 1 부족·초과되는 값 사용

### 7번 해설 - 인코딩 [코드작성]

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
data_preset = data_na.copy()
data_preset['District_Type'] = le.fit_transform(data_preset['District_Type'])

> 범주형 데이터를 숫자로 바꾸는 두 가지 방법 중 하나를 사용합니다.

### 8번 해설 - 데이터 분리 [코드작성]

In [ ]:
from sklearn.model_selection import train_test_split

X = data_preset.drop('Property_Class', axis=1)
y = data_preset['Property_Class']

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=7, stratify=y)

> train_test_split(X, y, ...) 은 X_train, X_valid, y_train, y_valid 순서로 반환합니다.

### 9번 해설 - 스케일링 [오류정정]

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

> [loop_control] 원래 코드는 조건 없이 항상 스케일링을 수행했지만, 버그 버전에서는 X_train의 길이가 0보다 클 경우에만 fit_transform이 실행되도록 if 문을 추가했습니다. 이는 실제 데이터가 비어있을 때 scaler 객체에 fit되지 않거나, 다른 로직에서 예상치 못한 None 값을 발생시켜 루프/조건 제어 착각을 유발할 수 있습니다.

### 10번 해설 - GridSearch 모델링 [코드작성]

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

gs_a = GridSearchCV(DecisionTreeClassifier(random_state=7), {'max_depth':[3,5,7]}, cv=3)
gs_a.fit(X_train_scaled, y_train)

gs_b = GridSearchCV(RandomForestClassifier(random_state=7), {'n_estimators':[50, 100, 200], 'max_depth':[3,5,7]}, cv=3)
gs_b.fit(X_train_scaled, y_train)

> GridSearchCV(estimator, param_grid, cv=...).fit(X_train, y_train) 형태로 탐색합니다.

### 11번 해설 - GridSearch 결과 확인 [결과값예측]

In [ ]:
200

> 제시된 후보 중 GridSearch가 고를 수 있는 최댓값을 묻는 문항입니다.

### 12번 해설 - 변수중요도 [오류정정]

In [ ]:
fi = pd.DataFrame({'feature': X_train.columns, 'importance': gs_b.best_estimator_.feature_importances_})
fi = fi.sort_values('importance', ascending=False)[:15]
sns.barplot(x='importance', y='feature', data=fi)
plt.show()

> [off_by_one] 원래 코드는 상위 15개의 특성을 선택하도록 설정되어 있었으나, 오프바이원 오류를 유발하기 위해 슬라이싱 범위를 16으로 변경했습니다. 이는 실제로는 16개의 특성(인덱스 0부터 15까지)을 선택하게 되어 의도했던 15개보다 하나 더 많은 데이터가 포함됩니다.

### 13번 해설 - 성능평가 [코드작성]

In [ ]:
from sklearn.metrics import f1_score

y_pred_a = gs_a.best_estimator_.predict(X_valid_scaled)
y_pred_b = gs_b.best_estimator_.predict(X_valid_scaled)

a_score = f1_score(y_valid, y_pred_a, average='macro')
b_score = f1_score(y_valid, y_pred_b, average='macro')
print(a_score, b_score)

> sklearn.metrics.f1_score 에 (실제값, 예측값) 순서로 인자를 넣습니다.